In [ ]:
#load "ExamEvaluation.fsx"
open ExamEvaluation
open System
open System.IO
open LectureScanning

# Evaluating an Exam Using FsLex

The scanner recognizes headers, maximum points, student names or matriculation numbers, and scores. A typed traversal computes grades; no state is attached dynamically to a lexer.

The grading and evaluation functions are defined visibly in `03-Exam-Evaluation.ipynb`. F# setup exports those teaching cells to `ExamEvaluation.fsx`; this file-based example imports them instead of duplicating the rules.

The function `load_data(fn)` takes one argument:
* `fn` is the name of a file

It returns the data that is stored in the specified files as a string.

In [ ]:
let loadData file = File.ReadAllText file

The result of a specific exam is stored in the file `exam.txt`:

In [ ]:
let data = loadData "exam.txt"
printfn "%s" data

This data show that there has been a exam on the subject *Advanced Witchcraft*
in the group *TINF09AI*.  Furthermore, the equation
```
MaxPoints = 75
```
shows that in order to achieve the best mark, **75** points would have been necessary.
    
There have been 5 different exercises in this exam. 
Our goal is to write a program that is able to compute the marks for all students.

## Imports

The compiled FsLex scanner is referenced as a .NET assembly. Its actions return typed `ExamToken` values; F# transformations compute the grades separately from scanning.

## Auxiliary Functions

The function `mark(max_points, points)` takes two arguments:
- `points`     is the number of points achieved by the student whose mark is to be computed.
- `max_points` is the number of points that need to be achieved in order to get the best mark of $1.0$.
  
It is assumed that the relation between the mark of an exam and the number of points achieved in this exam is mostly linear and that a student who has achieved $50\%$ of `lexer.max_points` points will get the mark $4.0$, while a student who has achieved  $100\%$ of `lexer.max_points` points will be marked as $1.0$.
However, the worst mark is $5.0$.  Therefore, if the mark would fall below that line, the `min` function below assures that it is lifted up to $5.0$.

In [ ]:
// Reuse the grading function from 03-Exam-Evaluation.ipynb.
mark 75 75

Lets test this function.

In [ ]:
for points in 0 .. 5 .. 80 do printfn "mark(75, %d) = %.1f" points (mark 75 points)

The function `percentage(max_points, points)` takes two arguments:
- `points`     is the number of points achieved by the student whose mark is to be computed.
- `max_points` is the number of points that need to be achieved in order to get the best mark of $1.0$.
  
It returns the percentage of points achieved.  Note that this percentage is never higher than `100%`, even if `points` is greater than `max_points`.

In [ ]:
// The canonical percentage function is shared with the preceding notebook.
percentage 75 75

In [ ]:
for points in 0 .. 5 .. 80 do printfn "percentage(75, %d) = %.0f%%" points (percentage 75 points)

## Token Declarations

`ExamToken` is a discriminated union: `Header`, `Maximum`, `Student`, `Points`, `Newline`, and `ExamEnd`. Every action has a known return type, and the evaluation match must account for every token case.

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
File.ReadAllText("Scanning/Types.fs") |> printfn "%s"

In [ ]:
printfn "%s" data

## Token Definitions

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

### The `HEADER` Token

`HEADER` matches class, group and exercise heading lines. The scanner returns `Header text`; the evaluator discards that descriptive metadata rather than counting its numbers as student scores.

In [ ]:
let rule23 = """| ("Class:" | "Group:" | "Exercise:") [^ '\n' '\r']* { Header(text lexbuf) }"""
printfn "%s" rule23

### The Token `MAXDEF`

`MAXDEF` recognizes `MaxPoints = n`. The FsLex action converts the decimal digits after `=` to an integer and returns `Maximum n`. Grade computation rejects a non-positive maximum.

In [ ]:
let rule25 = """| "MaxPoints" blank* '=' blank* natural { Maximum(int ((text lexbuf).Split('=').[1].Trim())) }"""
printfn "%s" rule25

### The Token Name

The scanner returns `Student name` for letters, spaces and hyphens followed by `:`. The evaluation traversal finalizes a previous row if needed and starts a new point total.

In [ ]:
let rule27 = """| digit digit digit digit digit digit digit ':' { Student((text lexbuf).TrimEnd(':')) }
| letter [ 'A'-'Z' 'a'-'z' ' ' '-']* ':' { Student((text lexbuf).TrimEnd(':').Trim()) }"""
printfn "%s" rule27

### The Token `MATRICULATION`

Seven decimal digits followed by a colon are returned as a `Student` token, retaining the matriculation number as a string so leading zeroes are preserved.

In [ ]:
let rule29 = """| digit digit digit digit digit digit digit ':' { Student((text lexbuf).TrimEnd(':')) }"""
printfn "%s" rule29

### The Token `NUMBER`

Natural-number tokens return `Points n`. During evaluation, scores are added only while a student is active; header and exercise metadata cannot accidentally contribute points.

In [ ]:
let rule31 = """| "MaxPoints" blank* '=' blank* natural { Maximum(int ((text lexbuf).Split('=').[1].Trim())) }
| natural { Points(int (text lexbuf)) }"""
printfn "%s" rule31

### The Token `IGNORE`

Blank lines produce newline tokens but have no effect when no student is active. Spaces, tabs, carriage returns and standalone hyphens are skipped by the generated scanner.

In [ ]:
let rule33 = """| ("Class:" | "Group:" | "Exercise:") [^ '\n' '\r']* { Header(text lexbuf) }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; Newline }"""
printfn "%s" rule33

### The Token `LINEBREAK`

Each newline updates the source position and returns `Newline`. The evaluation traversal finishes the current student row, calculating points, grade and percentage. It also flushes the final row at end of input, even if the file has no final newline.

In [ ]:
// ExamResult and evaluate come from the same exported notebook module.
// Keeping one owner prevents the two grading examples from drifting.

In [ ]:
let printResult row = printfn "%s has %d points and achieved %.1f: %.0f%%" row.Name row.Points row.Grade row.Percentage

---

We have now defined all of the tokens.  
Note that the scanner tries the regular expressions in the same order that we 
have used to define these tokens. 

---

### Ignoring Characters

The FsLex character class below skips spaces, tabs, carriage returns and standalone hyphens. A hyphen inside a student name remains part of its longer student-name match.

In [ ]:
let rule39 = """let blank = [' ' '\t']
| [' ' '\t' '\r' '-']+ { token lexbuf }"""
printfn "%s" rule39

### Error Handling

An unmatched character raises an error with its input line. Malformed exam scores are reported rather than silently producing a misleading grade.

In [ ]:
let rule41 = """| _ { failwithf "Illegal exam character %s at line %d" (text lexbuf) lexbuf.StartPos.Line }"""
printfn "%s" rule41

### Build-time scanner generation

The adjacent project generates scanner code before notebook execution. FsLex does not inspect notebook filenames.

### Generating the Scanner and Running It

The next line generates the scanner.

In [ ]:
let scan = Scanning.exam

Next, we feed an input string into the generated scanner.

In [ ]:
let examTokens = scan data

In order to scan the data that we provided in the last line, the function `scan` iterates
over all tokens generated by our scanner.

In [ ]:
let results = evaluate examTokens

Finally, we can run the scanner.

In [ ]:
results |> List.iter printResult

In [ ]:
// A typed record list replaces the pandas DataFrame.

In [ ]:
for index, row in List.indexed results do
    printfn "%d | %-20s | %2d | %.1f | %.0f%%" (index+1) row.Name row.Points row.Grade row.Percentage